# Suite SNP — Snapshots: a store written as a log of mutations, and read back

An initial snapshot ([Serialization](../../docs/SERIALIZATION.md)) is a store written by `Plain.ToPlain(store).Snapshot()`
as a log of mutations: its modules, then the schemas each defines, then the other named singletons, then the remaining
reference objects, each read through its visitor and written once in definition form (`exists`), its entries
(`insert`) once every object they link is. `Plain.FromPlain(store).Snapshot(log)` applies a log through the store's
builders. Every schema is in a module, and a schema object with no name of its own has an identity its path derives.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store are `Proxy` objects typed
loosely, so callbacks are annotated `(t: any)`; ints are `bigint`s, plain mappings are `Map`s, and a log written again
is compared as its JSON text.

In [ ]:
import { Builders as B, JSON as Json, Plain, Proxies, Stores } from "@mbse/schemas/Framework";
import { DecodeError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, same } from "./support.js";

/** crm.Person, who knows other persons since a year, and a singleton crm.Registry, built in `store`. */
function people(store: any): [any, any] {
  const [Int, Str] = [store.Schemas.OfInt, store.Schemas.OfStr];
  const Person = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Person")
      .ref(true)
      .properties([
        (t: any) => t.name("given_name").type([Str]),
        (t: any) => t.name("tags").type([store.Schemas.OfContainer((c: any) => c.index([Int]).element([Str]))]),
      ])
  ));
  const Knows = store.Schemas.OfRelation((r: any) => (
    r
      .name("crm.Knows")
      .links([
        (t: any) => t.name("knower"),
        (t: any) => t.name("known"),
      ])
      .properties([(t: any) => t.name("since").type([Int])])
      .uniques([["since"], ["knower", "known"]])
  ));
  store.Schemas.OfObject(Person, (o: any) => o.adjacencies([
    (t: any) => t.name("acquaintances").relation([Knows]).me("knower"),
    (t: any) => t.relation([Knows]).me("known"),
  ]));
  store.Schemas.OfObject((o: any) => o.name("crm.Registry").ref(true).singleton("the.Registry").properties([
    (t: any) => t.name("title").type([Str]),
  ]));
  return [Person, Knows];
}

function keys(log: any[], verb = "exists"): string[] {
  return log.filter((e) => e.has(verb))
    .map((e) => verb === "exists" ? e.get(verb).get("definition").get("key") : e.get(verb).get("relation").get("key"));
}

/** The properties each `exists` defines, by key. */
function definitions(log: any[]): Map<string, any> {
  return new Map(log.filter((e) => e.has("exists")).map((e) => [e.get("exists").get("definition").get("key"),
    e.get("exists").get("definition").get("properties")]));
}

/** A log as its JSON text, to compare two logs as Python compares them with `==`, in order. */
const text = (log: unknown) => Json.dumps(log as never);

## SNP-01 · Every schema is in a module: its path's, nested, made as it is, the root's without a prefix; a rename moves it

In [ ]:
{
  const store: any = new B.Store();
  const [Int, Str] = [store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()];
  const [Person] = people(store);
  const Loose = store.Schemas.OfObject((o: any) => o.name("Loose"));
  const Order = store.Schemas.OfObject((o: any) => o.name("crm.sales.Order"));
  const [root, crm, sales] = [store.module(), store.module("crm"), store.module("crm.sales")];
  assert(root.name === null && B.identity_of(root) === "." && same([...root.schemas.keys()], ["crm", "Loose"]));
  assert(same([...crm.schemas.keys()], ["Person", "Knows", "Registry", "sales"]) && crm.schemas.get("Person") === Person);
  assert(sales.schemas.get("Order") === Order && crm.schemas.get("sales") === sales && B.identity_of(sales) === "crm.sales");
  store.Schemas.OfObject(Order, (o: any) => o.name("crm.Order")); // renamed: moved
  assert(!sales.schemas.has("Order") && crm.schemas.get("Order") === Order);
  store.Schemas.OfModule(sales, (m: any) => m.name("crm.shop")); // a module renamed
  assert(store.module("crm.shop") === sales);
  const bad = store.Schemas.OfObject((o: any) => o.name("not a path"));
  assert([root, crm].every((module) => [...module.schemas.values()].every((held) => held !== bad))); // validation reports its name
  assert(store.framework(store.Schemas.OfObject.as_schema()) && store.framework(store.Schemas.OfInt.as_schema()) && !store.framework(Person));
  assert(store.held("Schemas.OfObject", "crm.Person") === Person && store.holds(Person) && !store.holds(new (Int.constructor as any)()));
  const extra = new B.ModuleBuilder(null, true, store).name("extra").schemas([Loose]).create(); // a schema linked with no name
  assert(same([...extra.schemas.keys()], ["Loose"]));
}

## SNP-02 · Identities a path derives: a member's from its owner and name, an unnamed type's from what first links it

In [ ]:
{
  const store: any = new B.Store();
  const [Int, Str] = [store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()];
  const [Person, Knows] = people(store);
  const tags = Person.properties.get("tags");
  assert(B.identity_of(tags) === "crm.Person.tags" && B.identity_of(tags.type) === "crm.Person.tags#type");
  assert(B.identity_of(Person.adjacencies.get("acquaintances")) === "crm.Person.acquaintances");
  assert(B.identity_of(Person.anonymous[0]) === "crm.Person#(crm.Knows.known)"); // anonymous: its relation and link
  assert(B.identity_of(Knows.entry.links.get("knower")) === "crm.Knows.knower");
  const Pending = store.Schemas.OfObject((o: any) => o.name("crm.Pending").ref(true).adjacencies([(t: any) => t.me("x")]));
  assert(![null, "crm.Pending#(None.x)"].includes(B.identity_of(Pending.anonymous[0]))); // no relation yet: it keeps its own
  const Unnamed = store.Schemas.OfObject((o: any) => o.name("crm.Unnamed").properties((adv: any) => adv.insert((e: any) => e.property((p: any) => p))));
  assert(B.identity_of(Unnamed.properties.get("")) === "crm.Unnamed#properties.0");
  const Kinds = store.Schemas.OfPredicate((p: any) => p.name("crm.Kinds").symbols([(t: any) => t.name("x")]).parameters([(t: any) => t.name("x")]));
  assert(B.identity_of(Kinds.parameters.get("x")) === "crm.Kinds.x" && B.identity_of(Kinds.symbols.get("x")) === "crm.Kinds.x"); // unique per kind
  const Twice = store.Schemas.OfUnion((u: any) => u.name("crm.Twice").branches([(t: any) => t.name("a").type([Int]), (t: any) => t.name("a").type([Str])]));
  assert(same(Twice.branches.map((b: any) => B.identity_of(b)), ["crm.Twice.a", "crm.Twice#branches.a"])); // invalid: told apart
  store.Schemas.OfProperty((p: any) => p.as_identity("crm.Taken.age").type([Int]));
  const Taken = store.Schemas.OfObject((o: any) => o.name("crm.Taken").properties([(t: any) => t.name("age").type([Int])]));
  assert(B.identity_of(Taken.properties.get("age")) === "crm.Taken#properties.age"); // its identity taken: told apart
  store.Schemas.OfContainer((c: any) => c.as_identity("crm.Held.x#type").index([Int]).element([Int]));
  raises(ValueError, () => store.Schemas.OfObject((o: any) => o.name("crm.Held").properties([
    (t: any) => t.name("x").type([store.Schemas.OfContainer((c: any) => c.index([Int]).element([Str]))]),
  ])), "a schema object with the identity 'crm.Held.x#type' is already in this store");
}

## SNP-03 · A snapshot: modules, schemas, named singletons, the rest; each object defined once, before what links it

In [ ]:
{
  const store: any = new B.Store();
  people(store);
  store.Schemas.OfObject((o: any) => o.name("crm.sales.Order").ref(true));
  const bob = store.crm.Person((c: any) => c.given_name("Bob"));
  const ada = store.crm.Person((c: any) => (
    c
      .given_name("Ada")
      .tags(["x", "y"])
      .acquaintances((adv: any) => (
        adv
          .insert((e: any) => e.known(bob).since(2020n))
      ))
  ));
  store.crm.Registry(store.singleton("the.Registry"), (r: any) => r.title("main"));
  const log = Plain.ToPlain(store).Snapshot() as any[];
  const defined = keys(log);
  assert(same(defined.slice(0, 3), ["Schemas.OfModule/.", "Schemas.OfModule/crm", "Schemas.OfModule/crm.sales"])); // the root, nested
  assert(defined.indexOf("Schemas.OfObject/crm.Person") < defined.indexOf("Schemas.OfProperty/crm.Person.given_name"));
  assert(defined.indexOf("crm.Registry/the.Registry") < defined.indexOf(`crm.Person/${bob.as_identity()}`)); // singletons first
  assert(defined.length === new Set(defined).size); // each once
  assert(!defined.some((key) => key.startsWith("Schemas.OfScalar/Schemas.OfStr") || key.startsWith("Schemas.OfObject/Schemas."))); // never written
  const written = definitions(log);
  assert(equal(written.get(`crm.Person/${ada.as_identity()}`), map({ given_name: "Ada", tags: ["x", "y"] }))); // untyped payloads
  assert(equal(written.get("crm.Registry/the.Registry"), map({}))); // a singleton exists in every store: its properties set after it
  assert(log.some((e) => equal(e, map({ as_property: { ref: { key: "crm.Registry/the.Registry" }, name: "title",
    mutations: [{ set: { value: "main" } }] } }))));
  assert(equal(written.get("Schemas.OfRelation/crm.Knows").get("uniques"), [["since"], ["knower", "known"]]));
  log.forEach((item, index) => { // an entry is inserted once the objects it links are defined
    if (!item.has("insert")) return;
    for (const entry of item.get("insert").get("entries")) {
      for (const member of entry.get("mutations")) {
        if (!member.has("as_link")) continue;
        const key = member.get("as_link").get("mutations")[0].get("set").get("ref").get("key");
        const before = defined.slice(0, log.slice(0, index).filter((e) => e.has("exists")).length);
        assert(before.includes(key) || (key.startsWith("Schemas.") && key.split("/")[1].startsWith("Schemas.")), key);
      }
    }
  });
  const knows = log.find((e) => e.has("insert") && e.get("insert").get("relation").get("key") === "Schemas.OfRelation/crm.Knows")
    .get("insert").get("entries");
  assert(equal(knows[0].get("mutations").filter((m: any) => m.has("as_property")),
    map([{ as_property: { name: "since", mutations: [{ set: { value: 2020n } }] } }])));
}

## SNP-04 · A log read back into another store, and written again the same; its objects held by the caller

In [ ]:
{
  const store: any = new B.Store();
  const [Person, Knows] = people(store);
  const bob = store.crm.Person((c: any) => c.given_name("Bob"));
  const ada = store.crm.Person((c: any) => c.given_name("Ada").acquaintances((adv: any) => adv.insert((e: any) => e.known(bob))));
  store.crm.Registry(store.singleton("the.Registry"), (r: any) => r.title("main"));
  const log = Plain.ToPlain(store).Snapshot();
  const back: any = new B.Store();
  const made = Plain.FromPlain(back).Snapshot(log);
  assert(text(Plain.ToPlain(back).Snapshot()) === text(log));
  assert(back.registered("crm.Person") !== Person && same([...back.registered("crm.Person").properties.keys()], ["given_name", "tags"]));
  const clauses = (uniques: readonly ReadonlySet<string>[]) => uniques.map((unique) => [...unique].sort()); // as Python compares frozensets
  assert(equal(clauses(back.registered("crm.Knows").uniques), clauses(Knows.uniques)) && back.singleton("the.Registry").title.get() === "main");
  const peopleBack = made.filter((m: any) => m instanceof Proxies.OfObject.Data && m.as_schema() === back.registered("crm.Person"));
  assert(same(peopleBack.map((p: any) => String(p.given_name)).sort(), ["Ada", "Bob"]));
  const again = Plain.FromPlain(back).Snapshot(log); // read again: every identity exists already
  assert(again.length === 0);
  Plain.FromPlain(back).Snapshot(map([{ as_property: { ref: { key: "crm.Registry/the.Registry" }, name: "title",
    mutations: [{ clear: {} }] } }]));
  assert(back.singleton("the.Registry").title.get() === null);
}

## SNP-05 · What a snapshot refuses: a store that is not a Builders.Store, a schema the store does not hold, a malformed log

In [ ]:
{
  raises(TypeError, () => Plain.ToPlain(new Stores.Combined(new B.Store())).Snapshot(), "a snapshot is of a Builders.Store"); // stores combined have no snapshot of their own
  raises(TypeError, () => Plain.FromPlain(new Stores.Combined(new B.Store())).Snapshot([]), "a snapshot is read into a Builders.Store");
  const setlike: any = new B.Store(); // a relation whose entries are a set: one empty clause, which a snapshot keeps
  setlike.Schemas.OfRelation((r: any) => r.name("Pairs").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques([[]]));
  const copied: any = new B.Store();
  Plain.FromPlain(copied).Snapshot(Plain.ToPlain(setlike).Snapshot());
  const uniques = copied.Pairs.as_schema().uniques;
  assert(uniques.length === 1 && uniques[0].size === 0); // read back, not dropped
  let store: any = new B.Store();
  const foreign = store.Schemas.OfObject((o: any) => o.name("crm.Foreign").properties([(t: any) => t.name("n").type([store.Schemas.OfInt.as_schema()])]));
  new B.OfProperty.Builder(foreign.properties.get("n")).type([new B.OfObject.Data()]).update(); // behind the store's back: no store builds it
  raises(ValueError, () => Plain.ToPlain(store).Snapshot(), "crm.Foreign.n, type, links _ObjectData, a schema the store does not hold");
  store = new B.Store();
  people(store);
  const Anonymous = store.Schemas.OfRelation((r: any) => r.name("crm.Anonymous").links([(t: any) => t.name("a"), (t: any) => t.name("b")]));
  store.Schemas.OfObject(store.registered("crm.Person"), (o: any) => o.adjacencies([
    (t: any) => t.relation([Anonymous]).me("a"),
    (t: any) => t.name("as_b").relation([Anonymous]).me("b"),
  ]));
  store.Schemas.OfRelation((r: any) => r.name("crm.Nowhere").links([(t: any) => t.name("a"), (t: any) => t.name("b")]));
  const bob = store.crm.Person((c: any) => c.given_name("Bob"));
  const ada = store.crm.Person((c: any) => c.given_name("Ada").as_b((adv: any) => adv.insert((e: any) => e.a(bob))));
  const log = Plain.ToPlain(store).Snapshot();
  const back: any = new B.Store();
  const made = Plain.FromPlain(back).Snapshot(log); // an entry added through its second link's adjacency: its first is anonymous
  assert(text(Plain.ToPlain(back).Snapshot()) === text(log));
  const person = `crm.Person/${bob.as_identity()}`;
  for (const [bad, says] of [
    [{}, "a log is a list, got dict"],
    [[[]], "a mutation is a mapping of one verb"],
    [[{ as_entries: {} }], "'as_entries' is not read in an initial snapshot"],
    [[{ exists: { definition: { key: "nokey" } } }], "a key is 'Module.Schema/ID', got 'nokey'"],
    [[{ exists: { definition: { key: "crm.Nobody/x" } } }], "no schema registered as 'crm.Nobody'"],
    [[{ exists: { definition: { key: "crm.Person/x", properties: { age: 1n } } } }], "crm.Person has no property 'age'"],
    [[{ as_property: { ref: { key: "crm.Registry/the.Registry" }, name: "nope", mutations: [] } }],
      "crm.Registry has no property 'nope'"],
    [[{ insert: { relation: { key: "Schemas.OfRelation/crm.Missing" }, entries: [] } }], "'Schemas.OfRelation/crm.Missing' is not in the store"],
    [[{ insert: { relation: { key: "Schemas.OfRelation/crm.Knows" }, entries: [{ mutations: [
      { as_link: { name: "knower", mutations: [{ set: { ref: { key: "crm.Knows/x" } } }] } }] }] } }], "'crm.Knows/x' is not in the store"],
    [[{ insert: { relation: { key: "Schemas.OfRelation/crm.Knows" }, entries: [{ mutations: [
      { as_property: { name: "when", mutations: [{ set: { value: 1n } }] } }] }] } }], "the relation has no property 'when'"],
    [[{ insert: { relation: { key: "Schemas.OfRelation/crm.Nowhere" }, entries: [{ mutations: [
      { as_link: { name: "a", mutations: [{ set: { ref: { key: person } } }] } }] }] } }],
      "no object an entry links declares an adjacency through which to add it"],
  ] as [unknown, string][]) {
    raises(DecodeError, () => Plain.FromPlain(back).Snapshot(map(bad)), says);
  }
  const ignored: Plain.Ignored[] = []; // a property a schema the store was made holding lacks is an extension's it lacks: ignored, not refused
  Plain.FromPlain(new B.Store()).Snapshot(map([{ exists: { definition: { key: "Schemas.OfObject/crm.X", properties: { nope: 1n } } } }]), ignored);
  assert(ignored.length === 1 && ignored[0].path === "$[0].exists.nope" && ignored[0].reason === "Schemas.OfObject has no property 'nope'");
}